<center>

# Deep Learning — Batch Normalization and Dropout in CNNs using Tensorflow

**Name:** Marrion Kiprop Cherop  
**Registration Number:** ST62/80971/2024  
**Programme:** Master of Science in Artificial Intelligence  
**Course:** CSA 809 — Deep Learning  
**Module:** Module 6 Normalization in CNN  
**Dataset:** Fashion-MNIST  
**Github Link:** https://github.com/MarrionKiprop/myOUK_Masters/blob/main/MLS-SEM3/CSA809-DL/CSA809_BatchNorm_Dropout_CNN_Marrion_Cherop.ipynb

<center>

## Introduction

This notebook examines what BatchNormalization and Dropout actually do to a convolutional network's training behaviour. The starting point is a baseline CNN with two convolutional blocks, no normalization and no regularization, trained on Fashion-MNIST. Against this baseline sits a deeper three-block CNN with BatchNormalization and Dropout applied after every convolutional layer, plus dropout before the output layer. Both models are trained under two conditions, first for five epochs to establish an initial reading, then for twenty-five epochs to see what each architecture does once given enough time to either converge or overfit. The question this notebook answers is not simply which model scores higher, but what each model's training and validation curves reveal about generalization, overfitting, and the actual cost and benefit of regularization at this scale of problem.


## Baseline CNN, 5 epochs

Two convolutional blocks (32 and 64 filters), same padding, no BatchNormalization, no Dropout. This establishes the reference point every later result is measured against.


In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.layers import BatchNormalization, Dropout
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.utils import to_categorical

# Load and preprocess data
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()
x_train = x_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0
x_test = x_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0

y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

# Build baseline CNN model
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1), padding='same'),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

# Compile the model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Train the model
model.fit(x_train, y_train,
          epochs=5,
          validation_data=(x_test, y_test))


29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 15s 6ms/step - accuracy: 0.8733 - loss: 0.3546 - val_accuracy: 0.9018 - val_loss: 0.2695
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9206 - loss: 0.2175 - val_accuracy: 0.9153 - val_loss: 0.2404
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9397 - loss: 0.1632 - val_accuracy: 0.9220 - val_loss: 0.2298
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9550 - loss: 0.1210 - val_accuracy: 0.9244 - val_loss: 0.2291
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9678 - loss: 0.0863 - val_accuracy: 0.9246 - val_loss: 0.2653



At five epochs the baseline reaches a train accuracy of 96.78% against a val_accuracy of 92.46%, a gap of 4.32 points. Val_loss follows a curve that matters more than the final number. It falls from 0.2695 to a low of 0.2291 at epoch 4, then rises again to 0.2653 at epoch 5. That uptick, arriving while train loss is still falling steeply (0.1210 to 0.0863), is the earliest visible sign that the network has already begun fitting noise in the training set rather than transferable structure. Five epochs is not enough to see where this leads, but the direction is already set. This model has no mechanism built in to resist that drift, so the question the next experiment has to answer is whether BatchNormalization and Dropout change that trajectory, or merely slow the network down without changing what it eventually does.


## BatchNorm + Dropout CNN, 5 epochs

Three convolutional blocks (32, 64, 128 filters), same padding preserving spatial dimensions through the network. BatchNormalization and Dropout(0.1) follow every convolutional layer, and Dropout(0.2) precedes the final dense output layer.


In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.layers import BatchNormalization, Dropout
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.utils import to_categorical

# Load and preprocess data
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()
x_train = x_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0
x_test = x_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0

y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

# Build deeper CNN model with BatchNorm and Dropout after every Conv2D layer
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1), padding='same'),
    layers.BatchNormalization(),
    layers.Dropout(0.1),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Dropout(0.1),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Dropout(0.1),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax')
])

# Compile the model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Train the model
model.fit(x_train, y_train,
          epochs=5,
          validation_data=(x_test, y_test))


Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - accuracy: 0.8426 - loss: 0.4369 - val_accuracy: 0.8828 - val_loss: 0.3299
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.8907 - loss: 0.2964 - val_accuracy: 0.8889 - val_loss: 0.3143
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9043 - loss: 0.2622 - val_accuracy: 0.8788 - val_loss: 0.3446
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9127 - loss: 0.2349 - val_accuracy: 0.8966 - val_loss: 0.3022
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9206 - loss: 0.2139 - val_accuracy: 0.9162 - val_loss: 0.2405


Reading only the epoch-5 and this model looks worse. Val_accuracy of 0.9162 against the baseline's 0.9246. That reading is misleading, because it ignores what each model is actually doing across the five epochs, not just where it lands. The train/val gap here is 92.06% train against 91.62% val, a difference of 0.44 points, against the baseline's 4.46-point gap at the same epoch count. Val_loss moves from 0.3446 down to 0.3022 at epoch 4 and stays essentially flat at 0.2405 by epoch 5, improving almost monotonically with no sign of the divergence the baseline already shows. The regularized model trains slower per epoch because BatchNormalization adds a normalization computation after every convolution and the extra conv block adds more parameters to update. That cost buys something specific, a network whose validation performance is not yet decoupling from its training performance. Five epochs is too short a window to know whether this model would eventually beat the baseline outright, which is the reason both models are extended to twenty-five epochs next.


## Baseline CNN, 25 epochs

The same baseline architecture as first experiment, extended to twenty-five epochs to observe what happens once the network is given enough time to either converge cleanly or overfit visibly.


In [3]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.layers import BatchNormalization, Dropout
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.utils import to_categorical

# Load and preprocess data
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()
x_train = x_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0
x_test = x_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0

y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

# Build baseline CNN model
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1), padding='same'),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

# Compile the model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Train the model
model.fit(x_train, y_train,
          epochs=25,
          validation_data=(x_test, y_test))


Epoch 1/25
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 12s 5ms/step - accuracy: 0.8748 - loss: 0.3489 - val_accuracy: 0.8992 - val_loss: 0.2774
Epoch 2/25
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9231 - loss: 0.2122 - val_accuracy: 0.9209 - val_loss: 0.2285
Epoch 3/25
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9405 - loss: 0.1594 - val_accuracy: 0.9227 - val_loss: 0.2221
Epoch 4/25
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9575 - loss: 0.1161 - val_accuracy: 0.9157 - val_loss: 0.2428
Epoch 5/25
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9690 - loss: 0.0834 - val_accuracy: 0.9260 - val_loss: 0.2458
Epoch 6/25
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9771 - loss: 0.0611 - val_accuracy: 0.9277 - val_loss: 0.2848
Epoch 7/25
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9853 - loss: 0.0412 - val_accuracy: 0.9260 - val_loss: 0.3319
Epoch 8/25
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9877 - loss: 0.0338 -

Extending the baseline to twenty-five epochs confirms what Experiment 1 only hinted at. Val_loss bottoms out early, at 0.2653 at epoch 5, and then climbs almost without interruption to 0.7649 by epoch 25, nearly tripling. Over the same span, train accuracy rises to 99.75% and train loss falls to 0.0077. This is overfitting in its clearest form. The network keeps finding ways to reduce training loss for twenty more epochs after it has already stopped learning anything transferable, and every one of those epochs makes the model worse for deployment even as the training log looks like it is improving. The best val_accuracy the baseline ever reaches, across the full 25 epochs, is 0.9277 at epoch 6. It never beats that number again. Left to run unchecked, this architecture actively degrades once past its optimal stopping point, which means reporting its final-epoch accuracy as the model's performance would be reporting the wrong number.


## BatchNorm + Dropout CNN, 25 epochs

The same regularized architecture as second experiment, extended to twenty-five epochs, run under the same conditions as Experiment 3 for a direct comparison at equal training length.


In [4]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.layers import BatchNormalization, Dropout
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.utils import to_categorical

# Load and preprocess data
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()
x_train = x_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0
x_test = x_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0

y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

# Build deeper CNN model with BatchNorm and Dropout after every Conv2D layer
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1), padding='same'),
    layers.BatchNormalization(),
    layers.Dropout(0.1),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Dropout(0.1),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Dropout(0.1),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax')
])

# Compile the model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Train the model
model.fit(x_train, y_train,
          epochs=25,
          validation_data=(x_test, y_test))


Epoch 1/25
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 18s 5ms/step - accuracy: 0.8464 - loss: 0.4243 - val_accuracy: 0.8596 - val_loss: 0.3728
Epoch 2/25
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.8930 - loss: 0.2911 - val_accuracy: 0.8903 - val_loss: 0.3058
Epoch 3/25
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9061 - loss: 0.2576 - val_accuracy: 0.8954 - val_loss: 0.2838
Epoch 4/25
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9129 - loss: 0.2353 - val_accuracy: 0.8766 - val_loss: 0.3336
Epoch 5/25
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9198 - loss: 0.2184 - val_accuracy: 0.8971 - val_loss: 0.3145
Epoch 6/25
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9255 - loss: 0.1991 - val_accuracy: 0.9051 - val_loss: 0.2809
Epoch 7/25
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9291 - loss: 0.1893 - val_accuracy: 0.9183 - val_loss: 0.2594
Epoch 8/25
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9345 - loss: 0.1748 -

The regularized model's twenty-five epoch run answers the question Experiment 2 left open. Val_loss reaches its best point of 0.2422 at epoch 8 and then stays contained in a 0.25 to 0.42 band through epoch 25, ending at 0.4231. Train accuracy at epoch 25 sits at 96.76%, well below the baseline's 99.75%, and that gap is the direct evidence of what regularization is doing: the model is declining to memorize the training set as tightly, and in exchange its validation behaviour stays stable instead of diverging. Its best val_accuracy across the full run is 0.9273 at epoch 24, which is higher than the baseline's best of 0.9277, and it reaches that peak while holding a train/val gap of about 4 points, against the baseline's 8-point gap at its own best epoch. This is the result that should anchor the conclusion: BatchNormalization and Dropout do not cost accuracy here, they buy a model whose peak performance is comparable or better and whose loss trajectory never turns against it.


## Conclusion

Four experiments, two architectures each run at two epoch counts, converge on one finding. Judged only on a shared, arbitrary epoch count, the plain baseline can look competitive or even superior, because it is free to fit the training set aggressively, including whatever noise sits inside it. Judged on each model's own optimal stopping point and on the shape of its validation loss curve across the full run, the picture reverses. The baseline peaks early and then deteriorates for later more epochs, with val_loss climbing from by over .50. The regularized model peaks later, at epoch 24, reaches a higher val_accuracy , and its val_loss never leaves a narrow, controlled band.

This distinction matters beyond this dataset. A model that overfits quickly is only safe to deploy if you know in advance exactly which epoch to stop at, which in practice means running validation-monitored early stopping, not guessing an epoch count. A model that overfits slowly and holds a small train/val gap is more forgiving of exactly this kind of uncertainty, and that forgiveness is what BatchNormalization and Dropout are actually built to deliver, not raw accuracy at a fixed epoch count. The practical recommendation from these four runs is to pair the regularized architecture with an EarlyStopping callback monitoring val_loss, so training halts automatically near the model's true optimum instead of relying on a fixed epoch count chosen without this evidence in hand.
